# Insmile AI — Fine-Tune Qwen2-VL-7B for Dental X-Ray Analysis

This notebook fine-tunes **Qwen2-VL-7B** on the **DENTEX** dataset using **QLoRA** (4-bit quantization + LoRA adapters) to fit within Google Colab's free T4 GPU (16GB VRAM).

**What this produces:** A LoRA adapter (~150MB) that makes the model significantly better at:
- Detecting dental caries, deep caries, periapical lesions, and impacted teeth
- Providing accurate FDI tooth numbers
- Generating precise bounding boxes
- Outputting structured JSON

## Instructions
1. Open this in Google Colab (GPU runtime → T4)
2. Run all cells in order
3. Training takes ~3-5 hours on a free T4
4. The adapter auto-uploads to HuggingFace when done

---

## 1. Setup — Install Dependencies

In [ ]:
%%capture
# Install all dependencies
!pip install "pillow<11" --quiet
!pip install -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 --quiet
!pip install -U transformers accelerate bitsandbytes peft trl datasets --quiet
!pip install -U qwen-vl-utils --quiet
!pip install -U huggingface_hub scipy kaggle --quiet

# Check if restart is needed (only on first run when Pillow was wrong)
import importlib, PIL
importlib.reload(PIL)
try:
    from PIL import ImageDraw
    print('All dependencies installed. No restart needed.')
except ImportError:
    print('Pillow updated — restarting runtime...')
    import os
    os.kill(os.getpid(), 9)

## 2. Configuration

In [ ]:
import os
from google.colab import userdata

# ============================================================
# CONFIGURATION — Edit these values
# ============================================================

# HuggingFace token — set in Colab secrets (Key icon on left sidebar → add HF_TOKEN)
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')

if not HF_TOKEN:
    raise ValueError(
        'HF_TOKEN not found! Add it in Colab: Left sidebar → Key icon → '
        'New secret → Name: HF_TOKEN, Value: your hf_... token'
    )

# Model — 2B fits on free T4 (15GB VRAM). 7B does NOT fit even in 4-bit.
BASE_MODEL = 'Qwen/Qwen2-VL-2B-Instruct'

# Where to save the fine-tuned adapter on HuggingFace
HF_REPO_NAME = 'joshuarebo/insmile-dental-vision-lora'

# Training hyperparameters
EPOCHS = 3
BATCH_SIZE = 1              # Limited by VRAM
GRAD_ACCUM_STEPS = 8       # Effective batch size = 1 * 8 = 8
LEARNING_RATE = 2e-4       # Standard for QLoRA
MAX_SEQ_LENGTH = 2048      # Max tokens per example
LORA_R = 16                # LoRA rank (higher = more capacity, more VRAM)
LORA_ALPHA = 32            # LoRA scaling factor
LORA_DROPOUT = 0.05

print(f'Model: {BASE_MODEL}')
print(f'Output repo: {HF_REPO_NAME}')
print(f'Epochs: {EPOCHS}, LR: {LEARNING_RATE}, LoRA rank: {LORA_R}')

## 3. Login to HuggingFace

In [ ]:
from huggingface_hub import login
login(token=HF_TOKEN)
print('Logged in to HuggingFace.')

## 4. Download Dental X-Ray Dataset

**Strategy:** Try DENTEX from HuggingFace first. If images fail to load, fall back to the Kaggle Dental Disease Panoramic dataset (YOLO format, ~2000 images, 31 classes).

**For Kaggle fallback:** Add your Kaggle API token as a Colab secret:
- Left sidebar → Key icon → New secret → Name: `KAGGLE_API_TOKEN`, Value: your token (starts with `KGAT_...`)
- Toggle "Notebook access" ON

In [ ]:
import json
import os
import subprocess
from pathlib import Path
from huggingface_hub import snapshot_download

DATASET_DIR = Path('/content/dental_data')
DATASET_DIR.mkdir(exist_ok=True)
DATASET_SOURCE = None

# ====================================================================
# STRATEGY 1: DENTEX from HuggingFace
# ====================================================================
print('='*60)
print('STRATEGY 1: Downloading DENTEX from HuggingFace...')
print('='*60)

DENTEX_DIR = DATASET_DIR / 'DENTEX'
try:
    snapshot_download(
        repo_id="ibrahimhamamci/DENTEX",
        repo_type="dataset",
        local_dir=str(DENTEX_DIR),
    )
    print('Download complete.')

    dentex_images = [p for p in DENTEX_DIR.rglob('*')
                     if p.is_file() and '.git' not in str(p)
                     and p.suffix.lower() in ('.png', '.jpg', '.jpeg')
                     and p.stat().st_size > 10000]

    if len(dentex_images) < 10:
        print(f'Only {len(dentex_images)} real images. Trying git-lfs pull...')
        subprocess.run(['apt-get', 'install', '-y', 'git-lfs'], capture_output=True)
        subprocess.run(['git', 'lfs', 'install'], cwd=str(DENTEX_DIR), capture_output=True)
        subprocess.run(['git', 'lfs', 'pull'], cwd=str(DENTEX_DIR),
                      capture_output=True, timeout=300)
        dentex_images = [p for p in DENTEX_DIR.rglob('*')
                         if p.is_file() and '.git' not in str(p)
                         and p.suffix.lower() in ('.png', '.jpg', '.jpeg')
                         and p.stat().st_size > 10000]

    if len(dentex_images) >= 10:
        DATASET_SOURCE = 'dentex'
        print(f'✅ DENTEX ready: {len(dentex_images)} images')
    else:
        print(f'❌ DENTEX failed: only {len(dentex_images)} real images')

except Exception as e:
    print(f'❌ DENTEX download failed: {e}')

# ====================================================================
# STRATEGY 2: Kaggle Dental Disease Panoramic (fallback)
# ====================================================================
if DATASET_SOURCE is None:
    print(f'\n{"="*60}')
    print('STRATEGY 2: Downloading from Kaggle...')
    print('='*60)

    KAGGLE_DIR = DATASET_DIR / 'kaggle_dental'
    KAGGLE_DIR.mkdir(exist_ok=True)

    try:
        from google.colab import userdata

        # Support BOTH new single-token and old username+key format
        kaggle_authenticated = False

        # Try new format first (KAGGLE_API_TOKEN)
        try:
            kaggle_token = userdata.get('KAGGLE_API_TOKEN')
            os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
            token_path = os.path.expanduser('~/.kaggle/access_token')
            with open(token_path, 'w') as f:
                f.write(kaggle_token)
            os.chmod(token_path, 0o600)
            kaggle_authenticated = True
            print('  Auth: Using KAGGLE_API_TOKEN')
        except Exception:
            pass

        # Try old format (KAGGLE_USERNAME + KAGGLE_KEY)
        if not kaggle_authenticated:
            try:
                ku = userdata.get('KAGGLE_USERNAME')
                kk = userdata.get('KAGGLE_KEY')
                os.environ['KAGGLE_USERNAME'] = ku
                os.environ['KAGGLE_KEY'] = kk
                kaggle_authenticated = True
                print('  Auth: Using KAGGLE_USERNAME + KAGGLE_KEY')
            except Exception:
                pass

        if not kaggle_authenticated:
            raise ValueError('No Kaggle credentials found in Colab secrets')

        # Download
        result = subprocess.run(
            ['kaggle', 'datasets', 'download', '-d',
             'lokisilvres/dental-disease-panoramic-detection-dataset',
             '--unzip', '-p', str(KAGGLE_DIR)],
            capture_output=True, text=True, timeout=600
        )
        print(f'  Output: {result.stdout[:200]}')
        if result.returncode != 0:
            print(f'  Error: {result.stderr[:300]}')

        kaggle_images = [p for p in KAGGLE_DIR.rglob('*')
                         if p.is_file() and p.suffix.lower() in ('.png', '.jpg', '.jpeg')
                         and p.stat().st_size > 10000]

        if len(kaggle_images) >= 10:
            DATASET_SOURCE = 'kaggle'
            print(f'✅ Kaggle dataset ready: {len(kaggle_images)} images')
        else:
            print(f'❌ Kaggle: only {len(kaggle_images)} images found')
            # Show what we got for debugging
            all_files = [p for p in KAGGLE_DIR.rglob('*') if p.is_file()]
            print(f'  Total files downloaded: {len(all_files)}')
            for p in all_files[:10]:
                print(f'    {p.relative_to(KAGGLE_DIR)} ({p.stat().st_size/1024:.1f}KB)')

    except Exception as e:
        print(f'❌ Kaggle failed: {e}')
        print('  Add KAGGLE_API_TOKEN to Colab secrets (Key icon → left sidebar)')

# ====================================================================
# REPORT
# ====================================================================
print(f'\n{"="*60}')
if DATASET_SOURCE == 'dentex':
    print(f'✅ Using DENTEX dataset ({len(dentex_images)} images)')
    ACTIVE_DIR = DENTEX_DIR
elif DATASET_SOURCE == 'kaggle':
    print(f'✅ Using Kaggle dental dataset ({len(kaggle_images)} images)')
    ACTIVE_DIR = KAGGLE_DIR
    print('\nDataset structure:')
    for item in sorted(KAGGLE_DIR.iterdir()):
        if item.is_dir():
            sub_count = len(list(item.rglob('*')))
            print(f'  📁 {item.name}/ ({sub_count} files)')
        else:
            print(f'  📄 {item.name} ({item.stat().st_size/1024:.1f}KB)')
else:
    raise RuntimeError(
        'No dataset available!\n'
        '  - DENTEX: Images stuck in Git LFS\n'
        '  - Kaggle: Add KAGGLE_API_TOKEN secret in Colab\n'
        '    (Get it from kaggle.com → Settings → API → Create New Token)'
    )

## 5. Parse DENTEX Annotations → Training Format

We convert the COCO-format annotations into instruction-tuning examples:
- **Input**: dental X-ray image + analysis prompt
- **Output**: structured JSON with findings, tooth numbers, severity, bounding boxes

In [ ]:
import json
import glob
from pathlib import Path
from PIL import Image

# DENTEX category mapping
DENTEX_CATEGORIES = {
    1: {'label': 'Caries', 'severity': 'moderate'},
    2: {'label': 'Deep caries', 'severity': 'severe'},
    3: {'label': 'Periapical lesion', 'severity': 'severe'},
    4: {'label': 'Impacted tooth', 'severity': 'moderate'},
}

# FDI quadrant mapping for tooth number estimation from bbox position
def estimate_fdi_from_bbox(bbox_norm, img_width, img_height):
    """Estimate FDI tooth number from bbox position on a panoramic X-ray.
    Panoramic X-rays are mirrored: patient's right is on image left.
    """
    cx = bbox_norm[0] + bbox_norm[2] / 2  # center x (0-1)
    cy = bbox_norm[1] + bbox_norm[3] / 2  # center y (0-1)
    
    # Determine quadrant
    is_upper = cy < 0.5
    is_right_side = cx < 0.5  # Image left = patient's right
    
    if is_upper and is_right_side:
        quadrant = 1
    elif is_upper and not is_right_side:
        quadrant = 2
    elif not is_upper and not is_right_side:
        quadrant = 3
    else:
        quadrant = 4
    
    # Estimate tooth position (1-8) based on distance from midline
    dist_from_center = abs(cx - 0.5) * 2  # 0 = midline, 1 = edge
    tooth_num = min(8, max(1, int(dist_from_center * 8) + 1))
    
    return f'{quadrant}{tooth_num}'


def parse_dentex_annotations(json_path, images_dir):
    """Parse COCO-format DENTEX annotations into training examples."""
    with open(json_path) as f:
        data = json.load(f)
    
    # Build image lookup
    images = {img['id']: img for img in data.get('images', [])}
    
    # Build category lookup
    categories = {}
    for cat in data.get('categories', []):
        cat_id = cat['id']
        cat_name = cat['name'].lower()
        if 'deep' in cat_name or 'caries' in cat_name and 'deep' in cat_name:
            categories[cat_id] = {'label': 'Deep caries', 'severity': 'severe'}
        elif 'caries' in cat_name:
            categories[cat_id] = {'label': 'Caries', 'severity': 'moderate'}
        elif 'periapical' in cat_name:
            categories[cat_id] = {'label': 'Periapical lesion', 'severity': 'severe'}
        elif 'impacted' in cat_name:
            categories[cat_id] = {'label': 'Impacted tooth', 'severity': 'moderate'}
        else:
            categories[cat_id] = {'label': cat['name'], 'severity': 'moderate'}
    
    # If no categories in file, use defaults
    if not categories:
        categories = DENTEX_CATEGORIES
    
    # Group annotations by image
    img_annotations = {}
    for ann in data.get('annotations', []):
        img_id = ann['image_id']
        if img_id not in img_annotations:
            img_annotations[img_id] = []
        img_annotations[img_id].append(ann)
    
    examples = []
    for img_id, anns in img_annotations.items():
        if img_id not in images:
            continue
        
        img_info = images[img_id]
        img_path = images_dir / img_info['file_name']
        if not img_path.exists():
            # Try common alternatives
            for alt in [images_dir / Path(img_info['file_name']).name,
                        images_dir.parent / 'xrays' / img_info['file_name']]:
                if alt.exists():
                    img_path = alt
                    break
            else:
                continue
        
        img_w = img_info.get('width', 1900)
        img_h = img_info.get('height', 950)
        
        findings = []
        for ann in anns:
            cat_id = ann.get('category_id', 1)
            cat = categories.get(cat_id, {'label': 'Abnormality', 'severity': 'moderate'})
            
            # COCO bbox format: [x, y, width, height] in pixels
            bbox = ann.get('bbox', [0, 0, 100, 100])
            bbox_norm = [
                round(bbox[0] / img_w, 4),
                round(bbox[1] / img_h, 4),
                round(bbox[2] / img_w, 4),
                round(bbox[3] / img_h, 4),
            ]
            
            # Estimate tooth number from position
            tooth = estimate_fdi_from_bbox(bbox_norm, img_w, img_h)
            
            findings.append({
                'label': f"{cat['label']} on tooth {tooth}",
                'tooth': tooth,
                'severity': cat['severity'],
                'confidence': 0.92,
                'bbox_norm': bbox_norm,
            })
        
        if not findings:
            continue
        
        # Build the target JSON
        target = {
            'findings': findings[:6],
            'overall': f"Panoramic X-ray showing {len(findings)} pathological finding{'s' if len(findings) > 1 else ''}: {', '.join(set(c['label'].split(' on ')[0] for c in findings[:4]))}.",
            'confidence': 0.88,
            'recommendations': generate_recommendations(findings),
            'image_quality': 'good',
        }
        
        examples.append({
            'image_path': str(img_path),
            'target_json': json.dumps(target, indent=None),
        })
    
    return examples


def generate_recommendations(findings):
    """Generate clinical recommendations based on findings."""
    recs = []
    labels = [f['label'].split(' on ')[0].lower() for f in findings]
    
    if 'deep caries' in labels:
        recs.append('Urgent: Root canal treatment or extraction may be needed for deep caries')
    if 'caries' in labels:
        recs.append('Composite or amalgam restoration recommended for carious lesions')
    if 'periapical lesion' in labels:
        recs.append('Periapical pathology detected — consider endodontic evaluation and vitality testing')
    if 'impacted tooth' in labels:
        recs.append('Surgical evaluation recommended for impacted tooth — assess proximity to IAN canal')
    if not recs:
        recs.append('Regular follow-up and preventive care recommended')
    
    recs.append('Patient education on oral hygiene and dietary habits')
    return recs[:4]


print('Annotation parser ready.')

In [ ]:
# Parse annotations into training examples
# Handles both DENTEX (COCO JSON) and Kaggle (YOLO txt) formats
all_examples = []

if DATASET_SOURCE == 'kaggle':
    # ================================================================
    # KAGGLE: Parse YOLO-format .txt annotations
    # Structure: images/ folder + labels/ folder + data.yaml
    # ================================================================
    print('Parsing Kaggle dataset (YOLO format)...')
    
    # Find classes from data.yaml or classes.txt
    class_names = []
    yaml_files = list(ACTIVE_DIR.rglob('data.yaml')) + list(ACTIVE_DIR.rglob('*.yaml'))
    for yf in yaml_files:
        try:
            with open(yf) as f:
                content = f.read()
            # Parse names from YAML (simple parser for the names list)
            if 'names:' in content:
                import re
                # Match names: ['class1', 'class2', ...] or names:\n  0: class1\n  1: class2
                bracket_match = re.search(r'names:\s*\[([^\]]+)\]', content)
                if bracket_match:
                    class_names = [n.strip().strip("'\"") for n in bracket_match.group(1).split(',')]
                else:
                    # YAML dict format: 0: classname
                    lines = content.split('names:')[1].split('\n')
                    for line in lines:
                        match = re.match(r'\s*\d+:\s*(.+)', line)
                        if match:
                            class_names.append(match.group(1).strip())
                        elif line.strip() and not line.strip().startswith('#') and ':' in line and not any(k in line for k in ['train:', 'val:', 'test:', 'nc:', 'path:']):
                            continue
                        elif class_names:
                            break
                print(f'  Found {len(class_names)} classes from {yf.name}')
                break
        except:
            continue
    
    if not class_names:
        # Default Kaggle dental classes
        class_names = ['Caries', 'Crown', 'Filling', 'Implant', 'Malaligned',
                       'Mandibular Canal', 'Missing teeth', 'Periapical lesion',
                       'Retained root', 'Root Canal Treatment', 'Root Piece',
                       'Impacted tooth', 'Maxillary sinus', 'Bone Loss',
                       'Fracture teeth', 'Permanent Teeth', 'Supra Eruption',
                       'TAD', 'Abutment', 'Attrition', 'Bone defect',
                       'Gingival former', 'Metal band', 'Orthodontic brackets',
                       'Permanent retainer', 'Post-core', 'Plating', 'Wire',
                       'Cyst', 'Root resorption', 'Primary teeth']
        print(f'  Using default 31 classes')
    
    print(f'  Classes: {class_names[:10]}...' if len(class_names) > 10 else f'  Classes: {class_names}')
    
    # Find all label .txt files
    txt_files = [f for f in ACTIVE_DIR.rglob('*.txt')
                 if f.stem != 'classes' and 'README' not in f.name
                 and f.parent.name in ('labels', 'train', 'val', 'test')]
    
    # Also check for txt files alongside images
    if not txt_files:
        txt_files = [f for f in ACTIVE_DIR.rglob('*.txt')
                     if f.stem != 'classes' and 'README' not in f.name
                     and f.stat().st_size > 0 and f.stat().st_size < 50000]
    
    print(f'  Found {len(txt_files)} label files')
    
    for txt_path in txt_files:
        # Find corresponding image
        img_path = None
        for ext in ['.jpg', '.jpeg', '.png', '.bmp']:
            # Same directory
            candidate = txt_path.with_suffix(ext)
            if candidate.exists():
                img_path = candidate
                break
            # Parallel images/ directory
            img_dir = txt_path.parent.parent / 'images' / txt_path.parent.name
            if img_dir.exists():
                candidate = img_dir / (txt_path.stem + ext)
                if candidate.exists():
                    img_path = candidate
                    break
            # images/ at same level as labels/
            img_dir2 = txt_path.parent.parent / 'images'
            if img_dir2.exists():
                candidate = img_dir2 / (txt_path.stem + ext)
                if candidate.exists():
                    img_path = candidate
                    break
        
        if img_path is None:
            continue
        
        # Parse YOLO annotations
        findings = []
        try:
            with open(txt_path) as f:
                lines = f.readlines()
            
            for line in lines:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                
                cls_id = int(parts[0])
                x_center = float(parts[1])
                y_center = float(parts[2])
                w = float(parts[3])
                h = float(parts[4])
                
                # Convert YOLO center format to top-left format
                x = x_center - w / 2
                y = y_center - h / 2
                bbox_norm = [round(x, 4), round(y, 4), round(w, 4), round(h, 4)]
                
                if cls_id < len(class_names):
                    cat_name = class_names[cls_id]
                else:
                    cat_name = f'Class {cls_id}'
                
                # Map severity
                cat_lower = cat_name.lower()
                if 'deep' in cat_lower or 'periapical' in cat_lower or 'lesion' in cat_lower or 'fracture' in cat_lower:
                    severity = 'severe'
                elif 'caries' in cat_lower or 'decay' in cat_lower or 'bone loss' in cat_lower:
                    severity = 'moderate'
                else:
                    severity = 'mild'
                
                tooth = estimate_fdi_from_bbox(bbox_norm, 1, 1)
                findings.append({
                    'label': f'{cat_name} on tooth {tooth}',
                    'tooth': tooth,
                    'severity': severity,
                    'confidence': 0.92,
                    'bbox_norm': bbox_norm,
                })
        except:
            continue
        
        if not findings:
            continue
        
        target = {
            'findings': findings[:6],
            'overall': f"Panoramic X-ray showing {len(findings)} finding{'s' if len(findings) > 1 else ''}: {', '.join(set(f['label'].split(' on ')[0] for f in findings[:6]))}.",
            'confidence': 0.88,
            'recommendations': generate_recommendations(findings),
            'image_quality': 'good',
        }
        
        all_examples.append({
            'image_path': str(img_path),
            'target_json': json.dumps(target, indent=None),
        })
    
    print(f'  ✅ Parsed {len(all_examples)} training examples from Kaggle YOLO data')

elif DATASET_SOURCE == 'dentex':
    # ================================================================
    # DENTEX: Parse COCO JSON annotations
    # ================================================================
    print('Parsing DENTEX dataset (COCO format)...')
    
    # Build image file index
    all_image_files = {}
    for p in ACTIVE_DIR.rglob('*'):
        if '.git' in str(p):
            continue
        if p.is_file() and p.suffix.lower() in ('.png', '.jpg', '.jpeg'):
            all_image_files[p.name] = p
            all_image_files[p.stem] = p
    print(f'  Indexed {len(all_image_files)} image files')
    
    # Parse all JSON files
    json_files = [f for f in ACTIVE_DIR.rglob('*.json') if '.git' not in str(f)]
    
    for jf in sorted(json_files):
        try:
            with open(jf) as f:
                data = json.load(f)
        except:
            continue
        
        if not isinstance(data, dict) or 'images' not in data or 'annotations' not in data:
            continue
        if len(data['annotations']) == 0:
            continue
        
        # Get categories (DENTEX uses categories_3 for diagnosis)
        categories = {}
        for key in ['categories_3', 'categories', 'categories_2', 'categories_1']:
            if key in data and data[key]:
                for cat in data[key]:
                    categories[cat['id']] = cat['name']
                break
        
        if not categories:
            categories = {1: 'Caries', 2: 'Deep Caries', 3: 'Periapical Lesion', 4: 'Impacted Tooth'}
        
        # Check for diagnosis categories
        cat_lower = ' '.join(v.lower() for v in categories.values())
        if not any(t in cat_lower for t in ['caries', 'periapical', 'impacted', 'lesion']):
            continue
        
        print(f'  📋 {jf.name}: {len(data["images"])} imgs, {len(data["annotations"])} anns')
        print(f'     Categories: {categories}')
        
        images_info = {img['id']: img for img in data['images']}
        img_anns = {}
        for ann in data['annotations']:
            img_id = ann['image_id']
            if img_id not in img_anns:
                img_anns[img_id] = []
            img_anns[img_id].append(ann)
        
        found = 0
        for img_id, anns in img_anns.items():
            if img_id not in images_info:
                continue
            img_info = images_info[img_id]
            fname = img_info['file_name']
            
            img_path = all_image_files.get(fname) or all_image_files.get(Path(fname).name) or all_image_files.get(Path(fname).stem)
            if img_path is None:
                continue
            
            found += 1
            img_w = img_info.get('width', 2900)
            img_h = img_info.get('height', 1250)
            
            findings = []
            for ann in anns:
                bbox = ann.get('bbox', [0, 0, 100, 100])
                cat_id = ann.get('category_id', ann.get('category_id_3', 0))
                cat_name = categories.get(cat_id, 'Abnormality')
                
                cat_l = cat_name.lower()
                if 'deep' in cat_l:
                    label, severity = 'Deep caries', 'severe'
                elif 'caries' in cat_l:
                    label, severity = 'Caries', 'moderate'
                elif 'periapical' in cat_l:
                    label, severity = 'Periapical lesion', 'severe'
                elif 'impacted' in cat_l:
                    label, severity = 'Impacted tooth', 'moderate'
                else:
                    label, severity = cat_name, 'moderate'
                
                bbox_norm = [round(bbox[0]/img_w, 4), round(bbox[1]/img_h, 4),
                             round(bbox[2]/img_w, 4), round(bbox[3]/img_h, 4)]
                tooth = estimate_fdi_from_bbox(bbox_norm, img_w, img_h)
                findings.append({
                    'label': f'{label} on tooth {tooth}', 'tooth': tooth,
                    'severity': severity, 'confidence': 0.92, 'bbox_norm': bbox_norm,
                })
            
            if not findings:
                continue
            
            target = {
                'findings': findings[:6],
                'overall': f"Panoramic X-ray showing {len(findings)} pathological finding{'s' if len(findings)>1 else ''}: {', '.join(set(f['label'].split(' on ')[0] for f in findings[:4]))}.",
                'confidence': 0.88,
                'recommendations': generate_recommendations(findings),
                'image_quality': 'good',
            }
            all_examples.append({
                'image_path': str(img_path),
                'target_json': json.dumps(target, indent=None),
            })
        
        print(f'     Matched {found} images')
    
    print(f'  ✅ Parsed {len(all_examples)} training examples from DENTEX')

# ====================================================================
# FINAL REPORT
# ====================================================================
print(f'\n{"="*60}')
if all_examples:
    print(f'✅ TOTAL TRAINING EXAMPLES: {len(all_examples)}')
    print(f'\nSample:')
    print(f'  Image: {all_examples[0]["image_path"]}')
    print(f'  Target: {all_examples[0]["target_json"][:300]}')
else:
    raise RuntimeError('No training examples parsed. Check output above.')

## 6. Build Training Dataset

Convert to the chat format that Qwen2-VL expects for instruction tuning.

In [ ]:
import random
from torch.utils.data import Dataset as TorchDataset

# System prompt (same as our production prompt, so the model learns our exact format)
SYSTEM_PROMPT = """You are an expert dental radiologist AI. Analyze the dental X-ray and return a JSON object with your findings.

Output format:
{"findings": [{"label": "specific finding", "tooth": "FDI number", "severity": "mild|moderate|severe", "confidence": 0.0-1.0, "bbox_norm": [x, y, w, h]}], "overall": "summary", "confidence": 0.0-1.0, "recommendations": ["action"], "image_quality": "good|fair|poor"}

Rules: bbox_norm values are 0.0-1.0 (normalized). Use FDI tooth numbering. Return JSON ONLY."""

USER_PROMPT = "Analyze this dental radiograph. Identify all visible pathology using FDI tooth numbering. Return ONLY the JSON object."

def build_conversation(example):
    """Build a chat conversation for training."""
    return {
        'messages': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {
                'role': 'user',
                'content': [
                    {'type': 'image', 'image': example['image_path']},
                    {'type': 'text', 'text': USER_PROMPT},
                ]
            },
            {'role': 'assistant', 'content': example['target_json']},
        ]
    }

# Simple dataset wrapper (avoids PyArrow mixed-type error with HF Dataset)
class ChatDataset(TorchDataset):
    def __init__(self, conversations):
        self.conversations = conversations
    def __len__(self):
        return len(self.conversations)
    def __getitem__(self, idx):
        return self.conversations[idx]

# Shuffle and split
random.seed(42)
random.shuffle(all_examples)

split_idx = max(1, int(len(all_examples) * 0.9))
train_examples = all_examples[:split_idx]
val_examples = all_examples[split_idx:] if split_idx < len(all_examples) else all_examples[-1:]

train_conversations = [build_conversation(ex) for ex in train_examples]
val_conversations = [build_conversation(ex) for ex in val_examples]

print(f'Training examples: {len(train_conversations)}')
print(f'Validation examples: {len(val_conversations)}')

# Create datasets (simple wrapper — no PyArrow serialization needed)
train_dataset = ChatDataset(train_conversations)
val_dataset = ChatDataset(val_conversations)

print(f'\nDataset ready.')
print(f'Sample user prompt: {train_dataset[0]["messages"][1]["content"][1]}')
print(f'Sample target (first 200 chars): {train_dataset[0]["messages"][2]["content"][:200]}')

## 7. Load Model with QLoRA (4-bit Quantization)

This loads the 7B model in 4-bit precision (~4GB VRAM) and attaches trainable LoRA adapters.

In [ ]:
import torch
import gc
from transformers import (
    AutoProcessor,
    Qwen2VLForConditionalGeneration,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Free memory from dataset loading
gc.collect()
torch.cuda.empty_cache()

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f'Loading {BASE_MODEL} in 4-bit...')
model = Qwen2VLForConditionalGeneration.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)

processor = AutoProcessor.from_pretrained(BASE_MODEL, trust_remote_code=True)

# Prepare for k-bit training
model = prepare_model_for_kbit_training(model)

print(f'Model loaded. Parameters: {model.num_parameters():,}')
print(f'GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB')

In [ ]:
# LoRA configuration — attach adapters to attention layers
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',  # Attention
        'gate_proj', 'up_proj', 'down_proj',       # MLP
    ],
)

model = get_peft_model(model, lora_config)

trainable, total = model.get_nb_trainable_parameters()
print(f'Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')
print(f'GPU memory after LoRA: {torch.cuda.memory_allocated() / 1e9:.1f} GB')

## 8. Training with SFTTrainer

Using TRL's SFTTrainer which handles the vision-language chat format automatically.

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir='/content/insmile-dental-checkpoints',
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=50,
    save_strategy='steps',
    save_steps=100,
    save_total_limit=3,
    bf16=True,
    gradient_checkpointing=True,
    dataset_text_field='',
    dataset_kwargs={'skip_prepare_dataset': True},
    dataloader_pin_memory=False,
    remove_unused_columns=False,
    report_to='none',
)

print('Training configuration ready.')
print(f'  Effective batch size: {BATCH_SIZE * GRAD_ACCUM_STEPS}')
print(f'  Total training steps: ~{len(train_conversations) * EPOCHS // (BATCH_SIZE * GRAD_ACCUM_STEPS)}')

In [ ]:
from functools import partial
from qwen_vl_utils import process_vision_info

def collate_fn(examples, processor):
    """Custom collator for vision-language training."""
    texts = []
    image_inputs = []
    
    for example in examples:
        messages = example['messages']
        text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        texts.append(text)
        
        images, videos = process_vision_info(messages)
        image_inputs.append(images)
    
    batch = processor(
        text=texts,
        images=image_inputs[0] if image_inputs[0] else None,
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        return_tensors='pt',
    )
    
    batch['labels'] = batch['input_ids'].clone()
    return batch

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=partial(collate_fn, processor=processor),
    processing_class=processor,
)

print('Trainer initialized. Ready to train.')

In [ ]:
# ============================================================
# TRAIN! This takes 3-5 hours on a T4 GPU.
# ============================================================
print('Starting training...')
print('='*60)

train_result = trainer.train()

print('='*60)
print('Training complete!')
print(f'  Loss: {train_result.training_loss:.4f}')
print(f'  Runtime: {train_result.metrics["train_runtime"]/3600:.1f} hours')
print(f'  Samples/sec: {train_result.metrics["train_samples_per_second"]:.2f}')

## 9. Save & Upload Adapter to HuggingFace

In [ ]:
# Save the LoRA adapter locally
ADAPTER_PATH = '/content/insmile-dental-adapter'
model.save_pretrained(ADAPTER_PATH)
processor.save_pretrained(ADAPTER_PATH)

print(f'Adapter saved to {ADAPTER_PATH}')

# Check adapter size
import os
total_size = sum(os.path.getsize(os.path.join(ADAPTER_PATH, f))
                 for f in os.listdir(ADAPTER_PATH)
                 if os.path.isfile(os.path.join(ADAPTER_PATH, f)))
print(f'Adapter size: {total_size / 1e6:.1f} MB')

In [ ]:
# Upload to HuggingFace Hub
from huggingface_hub import HfApi

api = HfApi()

# Create repo if it doesn't exist
try:
    api.create_repo(HF_REPO_NAME, private=True, exist_ok=True)
except Exception as e:
    print(f'Repo creation note: {e}')

# Upload adapter
api.upload_folder(
    folder_path=ADAPTER_PATH,
    repo_id=HF_REPO_NAME,
    commit_message='Insmile dental vision LoRA adapter - trained on DENTEX',
)

print(f'\nAdapter uploaded to: https://huggingface.co/{HF_REPO_NAME}')
print('\nYou can now use this adapter in your Insmile backend!')

## 10. Quick Validation — Test the Fine-Tuned Model

In [ ]:
# Test on a validation image
if val_examples:
    test_example = val_examples[0]
    test_image = Image.open(test_example['image_path']).convert('RGB')
    
    test_messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {
            'role': 'user',
            'content': [
                {'type': 'image', 'image': test_image},
                {'type': 'text', 'text': USER_PROMPT},
            ]
        },
    ]
    
    text = processor.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)
    images, _ = process_vision_info(test_messages)
    
    inputs = processor(
        text=[text], images=images, return_tensors='pt', padding=True
    ).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=1000, temperature=0.1)
    
    response = processor.decode(outputs[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)
    
    print('MODEL OUTPUT:')
    print(response[:500])
    print('\n---\nEXPECTED:')
    print(test_example['target_json'][:500])
else:
    print('No validation examples available for testing.')

## Done!

Your fine-tuned LoRA adapter is now on HuggingFace. Next steps:

1. **Deploy**: Host the model on RunPod Serverless or HuggingFace Inference Endpoints
2. **Integrate**: Update `server/src/services/openrouter.js` to call your self-hosted model
3. **Iterate**: As dentists use the app and correct findings, save those corrections as new training data

---
*Generated by Insmile AI training pipeline*